# 02 - Feature Engineering & Behavioral Signals
### Fraud Risk Intelligence & Detection Platform
This notebook covers:
1. Circadian temporal periodicity transformations (sine / cosine of hour-of-day).
2. Inter-arrival time delta calculations.
3. Non-linear monetary transformations & zero-amount indicators.
4. Latent PCA interaction norms and fraud risk polarities.
5. Fitting the unified `FeaturePipeline` on training data only to prevent data snooping.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.features import FeaturePipeline, compute_temporal_features
print("Imports loaded successfully.")


## 1. Inspect Temporal Cyclical Transformations
Transaction `Time` represents elapsed seconds over 48 hours. We convert seconds into a 24-hour cycle and compute sine and cosine projections.


In [ ]:
sample_df = pd.DataFrame({"Time": [0, 21600, 43200, 64800, 86400]})
sample_feat = compute_temporal_features(sample_df)
sample_feat[['Time', 'hour_of_day', 'hour_sin', 'hour_cos', 'time_delta_prev']]


## 2. Load Processed Splits & Fit FeaturePipeline
The pipeline must fit all statistics (means, standard deviations, scalers) strictly on `train_df`.


In [ ]:
train_df = pd.read_parquet("data/processed/train.parquet")
val_df = pd.read_parquet("data/processed/val.parquet")
test_df = pd.read_parquet("data/processed/test.parquet")

pipeline = FeaturePipeline()
pipeline.fit(train_df)

print(f"Fitted Feature Pipeline on {len(train_df):,} records.")
print(f"Total Output Features: {len(pipeline.feature_names_)}")
print("Sample Features:", pipeline.feature_names_[:15])


## 3. Transform Matrices & Verify Feature Distributions


In [ ]:
X_train = pipeline.transform(train_df)
X_val = pipeline.transform(val_df)
X_test = pipeline.transform(test_df)

print("X_train shape:", X_train.shape)
print("X_val shape:  ", X_val.shape)
print("X_test shape: ", X_test.shape)


## 4. Behavioral Deviation Features ($Z$-Score vs Baseline)


In [ ]:
behavior_cols = [c for c in X_train.columns if "amount_" in c or "v_latent" in c or "v_risk" in c or "log_amount" in c]
print("Engineered Behavioral Signals:")
print(X_train[behavior_cols].describe().round(3).T[['mean', 'std', 'min', '50%', 'max']])
